# CDF Hosted MQTT Broker — Step-by-step Setup

This notebook sets up a complete pipeline for ingesting real-time sensor data into CDF
using the **CDF-hosted MQTT broker** (beta).

```
mqtt_publisher.py
  │  publishes Cognite-format JSON over MQTT (TLS)
  ▼
CDF MQTT Broker  ← Source  (CDF hosts the broker)
  │
  │  Job subscribes to topic, applies Kuiper mapping
  ▼
Destination  (authenticated back into your CDF project)
  │
  └─► datapoint  { space: mqtt_demo, externalId: mqtt_demo_ts_000000, ... }
         │  auto-created on first datapoint
         ▼
      DM TimeSeries  instance_id=(mqtt_demo, mqtt_demo_ts_000000)
```

### Concepts

| Resource | Role |
|---|---|
| **Source** | Represents the MQTT broker. CDF provisions it and returns connection credentials. |
| **Destination** | Where the extractor writes data. Authenticated via a session nonce. |
| **Mapping** | A Kuiper expression that transforms each MQTT message into CDF outputs (nodes + datapoints). |
| **Job** | Links source → destination and defines the topic filter and which mapping to use. |

### How data flows

1. The publisher sends Cognite-format MQTT messages to the CDF-hosted MQTT broker.
2. The Kuiper mapping transforms each message into `datapoint` outputs with `space` set.
3. The extractor writes those datapoints to DM time series in the `mqtt_demo` space,
   **auto-creating** the node on the first datapoint for each `externalId`.
   Just change `num_timeseries` and re-run the publisher — no notebook re-run needed.

### Before you start

1. Copy `.env.example` to `.env` at the repo root and fill in your CDF project values.
2. Your CDF identity needs `hostedExtractors:READ` and `hostedExtractors:WRITE` capabilities.
3. Run **all cells top-to-bottom**. Re-running is safe — existing resources are detected and
   updated rather than duplicated.

> **API note:** The Hosted Extractors API is in **beta**. The Python SDK does not yet wrap these
> endpoints, so we use `client.post()` / `client.get()` directly with the header `cdf-version: beta`.

## Step 1 — Imports

In [ ]:
import sys
from pathlib import Path

from cognite.client.data_classes.data_modeling import ViewId
from dotenv import dotenv_values, set_key

repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "helpers").is_dir()
)
sys.path.insert(0, str(repo_root))
from helpers import SPACE, get_client

## Step 2 — Configuration

Edit the variables here to match your CDF project.

In [ ]:
# Cluster / project / OAuth IDs come from the repo-root .env via helpers.get_client().
# SPACE comes from helpers.SPACE.

# ── Hosted Extractor resource external IDs ────────────────────────────────────
DESTINATION_EXT_ID = "mqtt_demo_destination"
SOURCE_EXT_ID = "mqtt_demo_source"
MAPPING_EXT_ID = "mqtt_demo_mapping"
JOB_EXT_ID = "mqtt_demo_job"

# ── MQTT topic (must match topicFilter in the Job) ───────────────────────────
MQTT_TOPIC = "cdf/data"

# ── Optional: restrict datapoints to a CDF data set (None = no restriction) ──
TARGET_DATASET_ID: int | None = None

print(f"DM space          : {SPACE}")

## Step 3 — Connect to CDF and create the space

`get_client()` reads the repo-root `.env`. `LOGIN_FLOW` selects client-credentials or interactive browser login.

Creates the `mqtt_demo` data modeling space if it does not already exist. Teardown deletes it.

In [ ]:
from cognite.client.data_classes.data_modeling import SpaceApply

client = get_client(client_name="mqtt-broker-demo")
client.data_modeling.spaces.apply(SpaceApply(space=SPACE, name="MQTT broker demo"))
print(f"Space '{SPACE}' ready")

## Step 4 — API helpers

The Hosted Extractors API is in **beta** and requires the header `cdf-version: beta`.
These thin wrappers add that header and keep the rest of the notebook readable.

> **Delete pattern:** CDF bulk-deletes use `POST` to a `/delete` sub-path, not HTTP DELETE.

In [ ]:
BETA = {"cdf-version": "beta"}
HE = f"/api/v1/projects/{client.config.project}/hostedextractors"


def he_post(path: str, body: dict) -> dict:
    return client.post(f"{HE}/{path}", json=body, headers=BETA).json()


def he_get(path: str) -> dict:
    return client.get(f"{HE}/{path}", headers=BETA).json()


def he_delete(resource: str, external_ids: list[str]) -> None:
    """Bulk-delete hosted extractor resources (POST to the /delete sub-path)."""
    body = {"items": [{"externalId": eid} for eid in external_ids]}
    client.post(f"{HE}/{resource}/delete", json=body, headers=BETA)


def he_exists(resource: str, external_id: str) -> bool:
    """Return True if the resource with the given external ID already exists."""
    return any(
        item["externalId"] == external_id for item in he_get(resource).get("items", [])
    )


print("Helpers ready.")

## Step 5 — Create Destination

A **Destination** tells the extractor which CDF project to write data into.
It authenticates using a short-lived **session nonce** — a one-time token that must be
consumed immediately; it cannot be reused.

> **Important:** CDF consumes the nonce even on failed requests (e.g. 409 Duplicate).
> This cell always creates a **fresh nonce** right before the API call.

In [ ]:
dest_item = {
    "externalId": DESTINATION_EXT_ID,
    **({"targetDataSetId": TARGET_DATASET_ID} if TARGET_DATASET_ID else {}),
}

if not he_exists("destinations", DESTINATION_EXT_ID):
    session = client.iam.sessions.create()  # create nonce immediately before use
    resp = he_post(
        "destinations",
        {"items": [{**dest_item, "credentials": {"nonce": session.nonce}}]},
    )
    print(f"✓ Destination created  (session: {resp['items'][0]['sessionId']})")
else:
    # Refresh the nonce so the extractor session stays active
    session = (
        client.iam.sessions.create()
    )  # fresh nonce — previous one already consumed
    resp = he_post(
        "destinations/update",
        {
            "items": [
                {
                    "externalId": DESTINATION_EXT_ID,
                    "update": {"credentials": {"set": {"nonce": session.nonce}}},
                }
            ]
        },
    )
    print(
        f"✓ Destination exists — refreshed nonce  (session: {resp['items'][0]['sessionId']})"
    )

## Step 6 — Create Source (CDF-hosted MQTT broker)

A **Source** of type `mqtt_broker` tells CDF to provision a managed MQTT broker endpoint.
The creation response includes:
- **`host`** — the broker hostname your publisher connects to
- **`authentication.username`** and **`authentication.password`** — MQTT credentials

> **⚠ The password is only returned on first creation.** It is saved into the repo-root `.env` immediately,
> next to the CDF login settings. If you lose it, run the Teardown section at the bottom and re-create the source.

In [ ]:
env_path = repo_root / ".env"

if not he_exists("sources", SOURCE_EXT_ID):
    resp = he_post(
        "sources", {"items": [{"type": "mqtt_broker", "externalId": SOURCE_EXT_ID}]}
    )
    src = resp["items"][0]

    mqtt_host = src["host"]
    mqtt_port = src.get("port", 8883)
    mqtt_username = src["authentication"]["username"]
    mqtt_password = src["authentication"]["password"]

    # Merge into the repo-root .env. set_key updates these keys and leaves CDF login settings in place.
    set_key(env_path, "MQTT_HOST", mqtt_host)
    set_key(env_path, "MQTT_PORT", str(mqtt_port))
    set_key(env_path, "MQTT_USERNAME", mqtt_username)
    set_key(env_path, "MQTT_PASSWORD", mqtt_password)
    print("✓ Source created")
    print(f"  Host     : {mqtt_host}:{mqtt_port}")
    print(f"  Username : {mqtt_username}")
    print(f"  Password : {'*' * len(mqtt_password)}  (saved to {env_path.resolve()})")
else:
    src = next(
        s for s in he_get("sources")["items"] if s["externalId"] == SOURCE_EXT_ID
    )
    creds = dotenv_values(env_path)
    if not all(creds.get(k) for k in ("MQTT_HOST", "MQTT_USERNAME", "MQTT_PASSWORD")):
        print("⚠ Source exists but MQTT credentials are missing from .env — password cannot be recovered.")
        print("  Run the Teardown section, then re-run this notebook from Step 5.")
    else:
        print(f"✓ Source already exists — using credentials from {env_path}")
        print(f"  Host : {src['host']}:{src.get('port', 8883)}")

## Step 7 — Create Mapping

A **Mapping** is a [Kuiper](https://docs.cognite.com/cdf/integration/guides/extraction/hosted_extractors/kuiper_concepts)
expression that transforms each incoming MQTT message into CDF resources.

Our mapping receives the Cognite-format payload published by `mqtt_publisher.py`:

```json
{"timeseriesData": [{"externalId": "mqtt_demo_ts_000000", "items": [{"timestamp": 1234567890000, "value": 42.5}]}]}
```

For each datapoint, the mapping emits a `datapoint` output with **`space`** set.
This tells the extractor to write to the **Data Modeling time series** identified by
`instance_id = (space, externalId)`, **auto-creating** the node on first write if it doesn't exist.  
No classic time series are used.

In [ ]:
# Kuiper expression: for each (timeseries, datapoint) pair emit one datapoint output.
# The 'space' field routes each datapoint to the DM time series instance_id=(space, externalId).
# If the node doesn't exist yet the extractor creates it automatically on first write.
MAPPING_EXPRESSION = f"""
input.timeseriesData.flatmap(ts =>
    ts.items.map(dp => {{
        "type": "datapoint",
        "externalId": ts.externalId,
        "space": "{SPACE}",
        "timestamp": dp.timestamp,
        "value": dp.value
    }})
)
""".strip()

if not he_exists("mappings", MAPPING_EXT_ID):
    resp = he_post(
        "mappings",
        {
            "items": [
                {
                    "externalId": MAPPING_EXT_ID,
                    "mapping": {"expression": MAPPING_EXPRESSION},
                    "published": True,
                }
            ]
        },
    )
    print(f"✓ Mapping created: {MAPPING_EXT_ID}")
else:
    # Update the mapping expression in case it changed
    he_post(
        "mappings/update",
        {
            "items": [
                {
                    "externalId": MAPPING_EXT_ID,
                    "update": {"mapping": {"set": {"expression": MAPPING_EXPRESSION}}},
                }
            ]
        },
    )
    print(f"✓ Mapping already exists — expression updated: {MAPPING_EXT_ID}")

print(f"\nExpression:\n{MAPPING_EXPRESSION}")

## Step 8 — Create Job

A **Job** wires the source to the destination and specifies:
- `topicFilter` — which MQTT topic(s) to subscribe to
- `format` — the mapping to use to parse each message

We reference the custom mapping created above (`type: "custom"`).  
The job will have status `waiting` until the first message arrives, then `connected` / `running`.

In [ ]:
if not he_exists("jobs", JOB_EXT_ID):
    resp = he_post(
        "jobs",
        {
            "items": [
                {
                    "externalId": JOB_EXT_ID,
                    "sourceId": SOURCE_EXT_ID,
                    "destinationId": DESTINATION_EXT_ID,
                    "format": {"type": "custom", "mappingId": MAPPING_EXT_ID},
                    "config": {"topicFilter": MQTT_TOPIC},
                }
            ]
        },
    )
    job = resp["items"][0]
    print(f"✓ Job created  (status: {job['status']})")
else:
    job = next(j for j in he_get("jobs")["items"] if j["externalId"] == JOB_EXT_ID)
    print(f"✓ Job already exists  (status: {job['status']})")

print(f"  Topic filter : {job['config']['topicFilter']}")
print(f"  Format       : {job['format']}")

## Step 9 — Verify

List all hosted extractor resources to confirm everything is in order.

In [ ]:
print("=== Destinations ===")
for d in he_get("destinations")["items"]:
    print(f"  {d['externalId']}  (session: {d.get('sessionId')})")

print("\n=== Sources ===")
for s in he_get("sources")["items"]:
    print(
        f"  {s['externalId']}  type={s['type']}  host={s['host']}:{s.get('port', 8883)}"
    )

print("\n=== Mappings ===")
for m in he_get("mappings")["items"]:
    print(f"  {m['externalId']}")

print("\n=== Jobs ===")
for j in he_get("jobs")["items"]:
    print(
        f"  {j['externalId']}  status={j['status']}  topic={j['config']['topicFilter']}"
    )

print("\n=== DM time series ===")
from cognite.client.data_classes.data_modeling import (
    InstanceSort,
    filters as dm_filters,
)

TS_VIEW = ViewId("cdf_cdm", "CogniteTimeSeries", "v1")
ts_prefix_f = dm_filters.Prefix(["node", "externalId"], "mqtt_demo_ts_")
_kw = dict(
    instance_type="node", sources=[TS_VIEW], space=SPACE, limit=1, filter=ts_prefix_f
)
first_n = client.data_modeling.instances.list(
    **_kw, sort=[InstanceSort(property=["node", "externalId"], direction="ascending")]
)
last_n = client.data_modeling.instances.list(
    **_kw, sort=[InstanceSort(property=["node", "externalId"], direction="descending")]
)
if first_n:
    print(f"  First : {first_n[0].external_id}")
    print(f"  Last  : {last_n[0].external_id}")
else:
    print(f"  No 'mqtt_demo_ts_*' nodes in space '{SPACE}' yet")

## Step 10 — Run the publisher

From the repo root, run:

```bash
uv run python mqtt_demo/mqtt_publisher.py 10 1hz
uv run python mqtt_demo/mqtt_publisher.py 10 10hz
uv run python mqtt_demo/mqtt_publisher.py 1000 10hz
uv run python mqtt_demo/mqtt_publisher.py 100000 1hz
```

The Kuiper mapping emits each datapoint with `space` set to `helpers.SPACE` (`mqtt_demo` by default).  
On the **first datapoint** for a given `externalId` the extractor **auto-creates** the DM time series node  
`instance_id = (mqtt_demo, mqtt_demo_ts_NNNNNN)`.  
Just change the number and re-run the publisher — no notebook steps needed.

### What to check in Fusion

| Where | What to look for |
|---|---|
| Data management → Integrate → Extractors | `mqtt_demo_source` — status **Connected** |
| Data → Time series (filter: space = `mqtt_demo`) | `mqtt_demo_ts_000000` … appearing live |

---
## (Optional) Monitor — job status & metrics

Re-run this cell while the publisher is running. It takes ~5 seconds to complete  
because it samples the API twice to compute a live rate.

| Section | What it shows |
|---|---|
| **Live rate** | Delta between two API snapshots `SAMPLE_S` seconds apart — accurate immediately |
| **Current hour** | Cumulative counts for the current hourly bucket, rate = count / elapsed |
| **All-time total** | Sum across all fetched windows (shown once > 1 window exists) |
| write / transform errors | Shown only when > 0 |

In [ ]:
import datetime
import time

SAMPLE_S = 5  # seconds between the two snapshots used for the live rate


def _fetch_windows(limit=10):
    return (
        client.get(
            f"{HE}/jobs/metrics",
            params={"job": JOB_EXT_ID, "limit": limit},
            headers=BETA,
        )
        .json()
        .get("items", [])
    )


# ── Job status ────────────────────────────────────────────────────────────────
job = next(j for j in he_get("jobs")["items"] if j["externalId"] == JOB_EXT_ID)
status = job["status"]
icon = "✓" if status in ("running", "connected", "ok") else "✗"
print(f"{icon} {job['externalId']}  status={status}  target={job['targetStatus']}")
print()

# ── Live rate: two snapshots SAMPLE_S seconds apart ───────────────────────────
# The metrics API returns cumulative counters per hourly window.
# Taking a delta between two snapshots gives an instantaneous rate
# that is accurate from the very first second of publishing.
snap1 = _fetch_windows(limit=1)
time.sleep(SAMPLE_S)
snap2 = _fetch_windows(limit=1)
windows = _fetch_windows(limit=10)  # full history for the hourly section

if not windows:
    print("No metrics yet — start the publisher and re-run.")
else:
    # ── Live rate ─────────────────────────────────────────────────────────────
    if snap1 and snap2 and snap1[0].get("timestamp") == snap2[0].get("timestamp"):
        s1, s2 = snap1[0], snap2[0]
        d_msgs = s2.get("sourceMessages", 0) - s1.get("sourceMessages", 0)
        d_dps = s2.get("cdfUploadedValues", 0) - s1.get("cdfUploadedValues", 0)
        print(f"Live rate  (~{SAMPLE_S}s sample)")
        print(f"  msgs/s  {d_msgs / SAMPLE_S:>8.2f}")
        print(f"  dps/s   {d_dps / SAMPLE_S:>8.2f}")
        if d_msgs:
            print(f"  dps/msg {d_dps / d_msgs:>8.1f}    (≈ num_timeseries)")
    else:
        print("Live rate: window rolled over during sample — re-run.")
    print()

    # ── Hourly window ─────────────────────────────────────────────────────────
    # Window size from the gap between consecutive buckets; fall back to 1 h.
    win_s = (
        abs(windows[0]["timestamp"] - windows[1]["timestamp"]) / 1000
        if len(windows) >= 2
        else 3600
    )

    # Actual elapsed time inside the current (open) window.
    win_start_ms = windows[0]["timestamp"] - win_s * 1000
    elapsed_s = max(min((time.time() * 1000 - win_start_ms) / 1000, win_s), 1)

    def _fmt_window(w, dur_s):
        msgs = w.get("sourceMessages", 0)
        dps_u = w.get("cdfUploadedValues", 0)
        dps_i = w.get("cdfInputValues", 0)
        fails = w.get("cdfWriteFailures", 0)
        terrs = w.get("transformFailures", 0)
        skip = w.get("cdfSkippedValues", 0)
        print(f"  msgs/s  {msgs / dur_s:>8.2f}    received  : {msgs:>10,}")
        print(
            f"  dps/s   {dps_u / dur_s:>8.2f}    uploaded  : {dps_u:>10,}  / {dps_i:,} parsed"
        )
        if msgs:
            print(f"  dps/msg {dps_u / msgs:>8.1f}    (≈ num_timeseries)")
        for label, n in [
            ("write errors", fails),
            ("transform errors", terrs),
            ("skipped", skip),
        ]:
            if n:
                print(f"  ⚠ {label}: {n:,}")

    m0 = windows[0]
    t0 = datetime.datetime.fromtimestamp(
        m0["timestamp"] / 1000, tz=datetime.timezone.utc
    )
    print(
        f"Current hour  {t0.strftime('%Y-%m-%d %H:%M UTC')}  (elapsed {elapsed_s:.0f} s / {win_s / 60:.0f} min)"
    )
    _fmt_window(m0, elapsed_s)

    # ── All-time total ────────────────────────────────────────────────────────
    if len(windows) > 1:
        print()
        KEYS = (
            "sourceMessages",
            "cdfInputValues",
            "cdfUploadedValues",
            "cdfWriteFailures",
            "transformFailures",
            "cdfSkippedValues",
        )
        tot = {k: sum(w.get(k, 0) for w in windows) for k in KEYS}
        ts_ms = [w["timestamp"] for w in windows]
        span_s = (max(ts_ms) - min(ts_ms)) / 1000 + win_s
        print(f"All-time total  {len(windows)} window(s)  (~{span_s / 3600:.1f} h)")
        _fmt_window(tot, span_s)

### (Optional) Job logs — run when status shows errors

Fetches the most recent log entries for the job. Useful when `status` shows  
`transform_error`, `write_error`, or similar — the log messages will tell you exactly what went wrong.

If errors are persistent you can **restart the job** by running the restart cell below.

In [ ]:
import datetime

logs_resp = client.get(
    f"{HE}/jobs/logs",
    params={"job": JOB_EXT_ID, "limit": 20},
    headers=BETA,
).json()

entries = logs_resp.get("items", [])
if not entries:
    print("No log entries found.")
else:
    print(f"{len(entries)} most recent log entries (newest first):")
    print()
    for e in entries:
        ts = datetime.datetime.fromtimestamp(
            e.get("createdTime", 0) / 1000, tz=datetime.timezone.utc
        ).strftime("%H:%M:%S UTC")
        print(f"  [{ts}] {e.get('type', '?'):20s}  {e.get('message', '')}")

### (Optional) Check — DM time series nodes and latest datapoints

Directly queries Data Modeling instances in the space.  
Re-run while the publisher is running to confirm datapoints are flowing.

In [ ]:
import datetime
from cognite.client.data_classes.data_modeling import (
    NodeId,
    InstanceSort,
    filters as dm_filters,
)

TS_VIEW = ViewId("cdf_cdm", "CogniteTimeSeries", "v1")
ts_prefix_f = dm_filters.Prefix(["node", "externalId"], "mqtt_demo_ts_")
_sort_kw = dict(
    instance_type="node", sources=[TS_VIEW], space=SPACE, limit=1, filter=ts_prefix_f
)

first_n = client.data_modeling.instances.list(
    **_sort_kw,
    sort=[InstanceSort(property=["node", "externalId"], direction="ascending")],
)
last_n = client.data_modeling.instances.list(
    **_sort_kw,
    sort=[InstanceSort(property=["node", "externalId"], direction="descending")],
)

if not first_n:
    print(f"No nodes in space '{SPACE}' yet.")
    print("Start the publisher — nodes are auto-created on the first datapoint.")
else:
    first_id, last_id = first_n[0].external_id, last_n[0].external_id
    fmt = lambda ms: datetime.datetime.fromtimestamp(
        ms / 1000, tz=datetime.timezone.utc
    ).strftime("%H:%M:%S UTC")
    print(f"Nodes in space '{SPACE}'")
    print(f"  first : {first_id}")
    print(f"  last  : {last_id}")
    print()

    # Spot-check: latest datapoint on the first time series
    try:
        dps = client.time_series.data.retrieve(
            instance_id=NodeId(first_n[0].space, first_id),
            limit=1,
        )
        if dps and len(dps) > 0:
            print(
                f"Latest dp on {first_id}: value={dps[0].value}  @ {fmt(dps[0].timestamp)}"
            )
        else:
            print(f"{first_id} — no datapoints yet (publisher not running?)")
    except Exception as exc:
        print(f"Could not retrieve datapoints: {exc}")

### (Optional) Restart job

If the job is stuck in an error state, pause it and resume it to force a clean reconnect.

In [ ]:
def set_job_target(status: str) -> None:
    client.post(
        f"{HE}/jobs/update",
        json={
            "items": [
                {"externalId": JOB_EXT_ID, "update": {"targetStatus": {"set": status}}}
            ]
        },
        headers=BETA,
    ).raise_for_status()
    print(f"targetStatus → {status}")


import time as _time

set_job_target("paused")
_time.sleep(3)
set_job_target("running")
print("Done — re-run the monitor cell in a few seconds to check the new status.")

---
## (Optional) Teardown

Deletes all resources created by this notebook in the correct order:
1. **Job** (must go first — references source and destination)
2. **Source, Destination, Mapping**
3. **DM time series nodes** in `mqtt_demo` whose `external_id` starts with `mqtt_demo_ts_`
4. **Space** `mqtt_demo`
5. **MQTT keys** in the repo-root `.env` (CDF login settings are left in place)

Time series are identified by their **instance ID** (`space` + `external_id`) — no classic time series are used.

In [ ]:
from cognite.client.data_classes.data_modeling import ViewId

TS_PREFIX = "mqtt_demo_ts_"

# ── 1. Hosted extractor resources (job must be deleted first) ───────────────
for resource, ext_id in [
    ("jobs", JOB_EXT_ID),
    ("sources", SOURCE_EXT_ID),
    ("destinations", DESTINATION_EXT_ID),
    ("mappings", MAPPING_EXT_ID),
]:
    if he_exists(resource, ext_id):
        he_delete(resource, [ext_id])
        print(f"✓ Deleted {resource[:-1]:12s}: {ext_id}")
    else:
        print(f"  Skipped {resource[:-1]:12s}: {ext_id}  (not found)")

# ── 2. DM time series nodes (identified by instance_id: space + external_id) ─
from cognite.client.data_classes.data_modeling import filters as dm_filters

ts_nodes = client.data_modeling.instances.list(
    instance_type="node",
    sources=[ViewId("cdf_cdm", "CogniteTimeSeries", "v1")],
    space=SPACE,
    filter=dm_filters.Prefix(["node", "externalId"], TS_PREFIX),
    limit=None,
)
if ts_nodes:
    client.data_modeling.instances.delete(nodes=ts_nodes.as_ids())
    print(f"✓ Deleted {len(ts_nodes)} time series nodes from space '{SPACE}'")
    for n in ts_nodes:
        print(f"    ({n.space!r}, {n.external_id!r})")
else:
    print(f"  No '{TS_PREFIX}*' nodes found in space '{SPACE}'")

# ── 3. Space (after its instances are gone) ───────────────────────────────────
if client.data_modeling.spaces.retrieve(SPACE) is not None:
    client.data_modeling.spaces.delete(SPACE)
    print(f"✓ Deleted space '{SPACE}'")
else:
    print(f"  Skipped: space '{SPACE}'  (not found)")

# ── 4. MQTT credentials in the repo-root .env (CDF login settings stay) ──────
env_path = repo_root / ".env"
if env_path.exists():
    kept = []
    removed = False
    for line in env_path.read_text().splitlines():
        stripped = line.lstrip()
        key = (
            stripped.split("=", 1)[0].strip()
            if "=" in stripped and not stripped.startswith("#")
            else ""
        )
        if key.startswith("MQTT_"):
            removed = True
            continue
        kept.append(line)
    if removed:
        env_path.write_text("\n".join(kept).rstrip() + "\n")
        print("✓ Removed MQTT credentials from .env")
    else:
        print("  No MQTT credentials in .env")
else:
    print("  No .env file")

print("\nTeardown complete. Re-run from Step 5 to create fresh resources.")